# Langfuse metric experiments

Every metric has a separate explanation and test cell. Shared fixtures provide 10 RAG and 10 agent examples. These cells prepare labeled inputs; actual judge scores require the selected framework evaluator and model.

In [ ]:
import json
from pathlib import Path

experiment = {'framework': 'Langfuse', 'rag_metrics': ['faithfulness', 'answer_relevance', 'retrieval_relevance', 'context_recall', 'answer_correctness'], 'agent_metrics': ['task_success', 'tool_call_correctness', 'tool_argument_validity', 'step_efficiency', 'policy_adherence'], 'score_metadata': ['scorer_name', 'scorer_version', 'dataset_version', 'application_release'], 'status': 'plan_only: implement scorers and log versioned scores'}

def load_shared_cases(filename):
    for folder in (Path.cwd(), *Path.cwd().parents):
        for candidate in (folder / 'datasets' / filename, folder / 'opensource' / 'datasets' / filename):
            if candidate.is_file():
                return json.loads(candidate.read_text(encoding='utf-8'))
    raise FileNotFoundError(f'Cannot find shared dataset: {filename}')

rag_cases = load_shared_cases('rag_cases.json')
agent_cases = load_shared_cases('agent_cases.json')
assert len(rag_cases) >= 10 and len(agent_cases) >= 10
assert all({'question','contexts','response','reference','expected_context_relevance'} <= case.keys() for case in rag_cases)
assert all({'task','tool_trace','expected_tools','expected_outcome','final_response'} <= case.keys() for case in agent_cases)
print(f'Loaded {len(rag_cases)} RAG and {len(agent_cases)} agentic AI examples.')


## RAG metrics

The metric checks below are isolated so you can inspect and run them individually.

### Faithfulness

**What it measures:** Checks whether answer claims are supported by retrieved passages.

**Why/when to use it:** Use it to expose unsupported details; a judge compares response and contexts.

**Example fields:** question, contexts, response.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'faithfulness'
metric_fields = ['question', 'contexts', 'response']
metric_label = 'expected_grounded'
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in rag_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Answer Relevance

**What it measures:** Checks whether the response directly addresses the user question.

**Why/when to use it:** Use it to catch off-topic or evasive answers; relevance is distinct from correctness.

**Example fields:** question, response.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'answer_relevance'
metric_fields = ['question', 'response']
metric_label = 'expected_relevant'
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in rag_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Retrieval Relevance

**What it measures:** Checks whether retrieved passages are useful for the current query.

**Why/when to use it:** Use it to distinguish retrieval failures from generation failures.

**Example fields:** question, contexts, expected_context_relevance.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'retrieval_relevance'
metric_fields = ['question', 'contexts', 'expected_context_relevance']
metric_label = 'expected_context_relevance'
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in rag_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Context Recall

**What it measures:** Measures whether retrieval includes evidence needed to support the reference answer.

**Why/when to use it:** Use it to find missing evidence; requires a reference or required-evidence labels.

**Example fields:** question, contexts, reference.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'context_recall'
metric_fields = ['question', 'contexts', 'reference']
metric_label = None
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in rag_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Answer Correctness

**What it measures:** Compares generated answer with a trusted reference, including semantic factual agreement.

**Why/when to use it:** Use it when references or expert rubrics exist; it differs from grounding because a grounded answer can still omit required facts.

**Example fields:** question, response, reference.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'answer_correctness'
metric_fields = ['question', 'response', 'reference']
metric_label = None
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in rag_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


## Agentic AI metrics

The metric checks below are isolated so you can inspect and run them individually.

### Task Success

**What it measures:** Measures whether the agent accomplished the requested goal.

**Why/when to use it:** Use as end-to-end success; include cases where the agent falsely claims success.

**Example fields:** task, final_response, expected_outcome.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'task_success'
metric_fields = ['task', 'final_response', 'expected_outcome']
metric_label = 'expected_task_success'
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in agent_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Tool Call Correctness

**What it measures:** Checks that each tool call is appropriate and expected.

**Why/when to use it:** Use it to find wrong tool choices and omitted calls.

**Example fields:** task, tool_trace, expected_tools.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'tool_call_correctness'
metric_fields = ['task', 'tool_trace', 'expected_tools']
metric_label = 'expected_tool_correctness'
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in agent_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Tool Argument Validity

**What it measures:** Validates tool arguments against required values, types, and constraints.

**Why/when to use it:** Use schema and task checks to catch invalid inputs before tool execution.

**Example fields:** task, tool_trace.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'tool_argument_validity'
metric_fields = ['task', 'tool_trace']
metric_label = 'expected_argument_correctness'
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in agent_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Step Efficiency

**What it measures:** Checks whether the agent avoids needless steps, retries, and duplicate calls.

**Why/when to use it:** Use it to control latency/cost without rewarding skipped safety checks.

**Example fields:** task, tool_trace, max_steps.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'step_efficiency'
metric_fields = ['task', 'tool_trace', 'max_steps']
metric_label = None
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in agent_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Policy Adherence

**What it measures:** Measures whether the agent follows its assigned operational policy.

**Why/when to use it:** Use it to catch unsafe shortcuts and prohibited calls, including in successful runs.

**Example fields:** task, tool_trace, final_response.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'policy_adherence'
metric_fields = ['task', 'tool_trace', 'final_response']
metric_label = 'expected_policy_compliance'
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in agent_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})
